# Phase 5: Backtesting, Validation & Research Hygiene  
## Notebook 05_06 — Performance Haircut and Deflated Sharpe

### Research question

How do we adjust backtest performance for multiple testing, non-normal returns, short sample length, selection bias, and the fact that the best-looking Sharpe ratio is often the winner of a noise contest?

This notebook follows:

```text
05_01_vectorized_backtest_engine
05_02_event_driven_backtest_engine
05_03_transaction_costs_slippage_latency
05_04_walk_forward_optimization
05_05_bayesian_strategy_calibration
```

The previous notebook introduced Bayesian calibration. This notebook focuses on the researcher's most dangerous metric:

$$
\widehat{Sharpe}
$$

A high Sharpe can arise from:

1. genuine alpha;
2. lucky noise;
3. too few observations;
4. non-normal returns;
5. many parameter trials;
6. hidden researcher degrees of freedom;
7. survivorship and selection bias;
8. unreported failed experiments.

It covers:

1. Sharpe ratio estimation error;
2. skew and kurtosis corrections;
3. Probabilistic Sharpe Ratio;
4. Minimum Track Record Length;
5. multiple testing;
6. expected maximum Sharpe under noise;
7. Deflated Sharpe Ratio;
8. performance haircuts;
9. parameter-grid overfit simulation;
10. false discovery intuition;
11. walk-forward degradation haircut;
12. non-normal strategy returns;
13. reporting conservative Sharpe;
14. governance flags;
15. saved outputs and manifest.

The key idea:

> A backtest Sharpe is not a trophy. It is noisy evidence that must be deflated for non-normality, short history, and the number of strategies tried.

## 1. Naive Sharpe ratio

Daily Sharpe annualised:

$$
\widehat{SR} = \frac{\bar r}{s_r}\sqrt{252}
$$

This estimate is noisy.

Even if true Sharpe is zero, testing many strategies creates a high maximum observed Sharpe.

That is selection bias.

If you test 500 variants and report only the best one, the best Sharpe is not an unbiased estimate of alpha.

## 2. Probabilistic Sharpe Ratio

The Probabilistic Sharpe Ratio estimates:

$$
P(SR > SR^*)
$$

given observed Sharpe, sample length, skew, and kurtosis.

A commonly used non-normal adjustment is:

$$
\begin{aligned}
PSR &= \Phi \Big( \frac{(\widehat{SR}-SR^*)\sqrt{n-1}} {\sqrt{1-\gamma_3\widehat{SR}+\frac{\gamma_4-1}{4}\widehat{SR}^2}} \Big)
\end{aligned}
$$

where:

- $n$ is number of observations;
- $\gamma_3$ is skewness;
- $\gamma_4$ is kurtosis, not excess kurtosis;
- $SR^*$ is a benchmark Sharpe.

If $SR^*=0$, PSR measures probability the true Sharpe is positive.

## 3. Deflated Sharpe Ratio

The Deflated Sharpe Ratio adjusts the benchmark Sharpe upward to account for multiple trials.

Instead of testing:

$$
SR > 0
$$

it tests whether the observed Sharpe beats the Sharpe you might expect from the best of many noisy trials.

A simplified workflow:

1. estimate observed Sharpe for all tried strategies;
2. estimate the expected maximum Sharpe under multiple testing;
3. use that as $SR^*$;
4. compute PSR against that higher benchmark.

This produces the Deflated Sharpe Ratio probability.

A strategy can have a high Sharpe and still have a poor DSR if many similar strategies were tested.

## 4. Performance haircut

A practical research report often includes a haircut:

$$
\begin{aligned}
SR_{haircut} &= SR_{observed} \\
&\quad - penalty
\end{aligned}
$$

Possible penalties:

- multiple testing penalty;
- non-normality penalty;
- short sample penalty;
- turnover/cost uncertainty penalty;
- walk-forward degradation penalty.

This notebook builds a transparent haircut scorecard rather than pretending one formula captures every research risk.

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from statistics import NormalDist
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class DeflatedSharpeConfig:
    n_days: int = 1_750
    annualisation: int = 252
    n_strategy_trials: int = 360
    n_true_alpha_strategies: int = 12
    seed: int = 42
    cvar_alpha: float = 0.95
    benchmark_sharpe: float = 0.0
    sharpe_threshold: float = 1.0
    psr_confidence: float = 0.95
    walk_forward_train: int = 504
    walk_forward_test: int = 126
    walk_forward_step: int = 126
    multiple_testing_temperature: float = 1.0

config = DeflatedSharpeConfig()
normal = NormalDist()
config

## 6. Utility functions

We need:

- annualised Sharpe;
- skew;
- kurtosis;
- PSR;
- expected maximum Sharpe under multiple trials;
- DSR;
- performance haircut.

In [ ]:
def annualised_sharpe(returns, annualisation=252):
    r = pd.Series(returns).dropna()
    if len(r) < 2 or r.std(ddof=1) == 0:
        return np.nan
    return float(r.mean() / r.std(ddof=1) * np.sqrt(annualisation))

def annualised_return(returns, annualisation=252):
    r = pd.Series(returns).dropna()
    if len(r) == 0:
        return np.nan
    return float((1 + r).prod() ** (annualisation / len(r)) - 1)

def annualised_vol(returns, annualisation=252):
    r = pd.Series(returns).dropna()
    return float(r.std(ddof=1) * np.sqrt(annualisation)) if len(r) > 1 else np.nan

def max_drawdown(nav):
    dd = nav / nav.cummax() - 1.0
    return dd, float(dd.min())

def historical_var_cvar(losses, alpha):
    losses = pd.Series(losses).dropna()
    var = losses.quantile(alpha)
    tail = losses[losses >= var]
    return float(var), float(tail.mean()) if len(tail) else np.nan

def skewness(x):
    return float(pd.Series(x).dropna().skew())

def kurtosis_non_excess(x):
    return float(pd.Series(x).dropna().kurtosis() + 3.0)

def probabilistic_sharpe_ratio(sr_observed, sr_benchmark, n_obs, skew, kurtosis):
    if not np.isfinite(sr_observed) or n_obs <= 2:
        return np.nan

    denominator = 1.0 - skew * sr_observed + ((kurtosis - 1.0) / 4.0) * sr_observed**2
    denominator = max(denominator, 1e-12)

    z = (sr_observed - sr_benchmark) * np.sqrt(n_obs - 1.0) / np.sqrt(denominator)
    return float(normal.cdf(z))

def expected_max_standard_normal(n_trials):
    # Approximation used in Deflated Sharpe discussions.
    # gamma is Euler-Mascheroni constant.
    if n_trials <= 1:
        return 0.0
    gamma = 0.5772156649
    term1 = (1.0 - gamma) * normal.inv_cdf(1.0 - 1.0 / n_trials)
    term2 = gamma * normal.inv_cdf(1.0 - np.exp(-1.0) / n_trials)
    return float(term1 + term2)

def sharpe_standard_error(sr_observed, n_obs, skew, kurtosis):
    if n_obs <= 2:
        return np.nan
    denominator = n_obs - 1.0
    variance = (1.0 - skew * sr_observed + ((kurtosis - 1.0) / 4.0) * sr_observed**2) / denominator
    return float(np.sqrt(max(variance, 1e-12)))

def expected_max_sharpe_under_trials(sharpe_values, n_obs):
    sr = pd.Series(sharpe_values).replace([np.inf, -np.inf], np.nan).dropna()
    if len(sr) == 0:
        return np.nan

    sr_std = sr.std(ddof=1)
    if sr_std == 0 or not np.isfinite(sr_std):
        sr_std = 1.0 / np.sqrt(max(n_obs - 1, 1))

    emz = expected_max_standard_normal(len(sr))
    return float(sr.mean() + sr_std * emz)

def deflated_sharpe_probability(sr_observed, sr_deflated_benchmark, n_obs, skew, kurtosis):
    return probabilistic_sharpe_ratio(sr_observed, sr_deflated_benchmark, n_obs, skew, kurtosis)

def minimum_track_record_length(sr_observed, sr_benchmark, skew, kurtosis, confidence=0.95):
    if not np.isfinite(sr_observed) or sr_observed <= sr_benchmark:
        return np.inf

    z = normal.inv_cdf(confidence)
    numerator = 1.0 - skew * sr_observed + ((kurtosis - 1.0) / 4.0) * sr_observed**2
    numerator = max(numerator, 1e-12)
    return float(1.0 + numerator * (z / (sr_observed - sr_benchmark)) ** 2)

## 7. Simulate many strategy trials

We simulate many candidate strategies.

Most have zero true alpha.

A small subset has weak positive true alpha.

Returns have:

- market factor exposure;
- idiosyncratic noise;
- occasional left-tail shocks;
- skew and kurtosis.

This mimics a research library where many ideas are tried, but only a few have real signal.

In [ ]:
def simulate_strategy_trials(config: DeflatedSharpeConfig):
    rng = np.random.default_rng(config.seed)
    dates = pd.bdate_range("2016-01-01", periods=config.n_days)

    n = config.n_strategy_trials
    strategy_ids = [f"strategy_{i:03d}" for i in range(n)]

    market = rng.standard_t(df=6, size=config.n_days) * 0.010
    vol_regime = np.zeros(config.n_days, dtype=int)
    stress_type = np.array(["normal"] * config.n_days, dtype=object)

    for t in range(1, config.n_days):
        if vol_regime[t - 1] == 0:
            vol_regime[t] = rng.choice([0, 1], p=[0.985, 0.015])
        else:
            vol_regime[t] = rng.choice([0, 1], p=[0.060, 0.940])

    true_alpha_daily = np.zeros(n)
    alpha_indices = rng.choice(n, size=config.n_true_alpha_strategies, replace=False)
    true_alpha_daily[alpha_indices] = rng.uniform(0.00008, 0.00025, size=config.n_true_alpha_strategies)

    beta_to_market = rng.normal(0.15, 0.35, size=n)
    idio_vol = rng.uniform(0.006, 0.018, size=n)
    tail_loading = rng.uniform(0.0, 0.012, size=n)
    turnover = rng.uniform(0.05, 0.80, size=n)

    returns = np.zeros((config.n_days, n))

    for t in range(config.n_days):
        vol_mult = 1.0 if vol_regime[t] == 0 else 2.4

        stress = 0.0
        u = rng.uniform()
        if u < 0.010:
            stress_type[t] = "left_tail_event"
            stress = -rng.uniform(0.020, 0.080)
        elif u < 0.016:
            stress_type[t] = "right_tail_event"
            stress = rng.uniform(0.015, 0.050)

        common_noise = beta_to_market * market[t]
        idio = rng.standard_t(df=5, size=n) * np.sqrt((5 - 2) / 5) * idio_vol * vol_mult
        tail = tail_loading * stress

        gross = true_alpha_daily + common_noise + idio + tail

        # Trading cost drag increases with turnover.
        cost_drag = turnover * rng.uniform(0.000005, 0.000035, size=n)
        returns[t] = gross - cost_drag

    returns_df = pd.DataFrame(returns, index=dates, columns=strategy_ids)

    metadata = pd.DataFrame({
        "strategy_id": strategy_ids,
        "true_alpha_daily": true_alpha_daily,
        "true_alpha_ann": true_alpha_daily * config.annualisation,
        "has_true_alpha": true_alpha_daily > 0,
        "beta_to_market": beta_to_market,
        "idio_vol_daily": idio_vol,
        "tail_loading": tail_loading,
        "turnover_assumption": turnover,
    })

    regime_info = pd.DataFrame({
        "market_return": market,
        "vol_regime": vol_regime,
        "stress_type": stress_type,
    }, index=dates)

    return returns_df, metadata, regime_info

strategy_returns, strategy_metadata, regime_info = simulate_strategy_trials(config)

strategy_returns.head(), strategy_metadata.head()

In [ ]:
plt.figure(figsize=(12, 6))
for col in strategy_returns.columns[:10]:
    plt.plot(strategy_returns.index, (1 + strategy_returns[col]).cumprod(), alpha=0.65, label=col)
plt.title("Example Strategy Trial Equity Curves")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend(ncol=2)
plt.show()

## 8. Compute naive performance for all strategies

This is what a careless researcher might sort by.

The top Sharpe will look impressive because there are many trials.

In [ ]:
def strategy_performance_table(strategy_returns, config):
    rows = []

    for col in strategy_returns.columns:
        r = strategy_returns[col].dropna()
        nav = (1 + r).cumprod()
        _, mdd = max_drawdown(nav)
        var, cvar = historical_var_cvar(-r, config.cvar_alpha)

        rows.append({
            "strategy_id": col,
            "n_obs": len(r),
            "annualised_return": annualised_return(r, config.annualisation),
            "annualised_vol": annualised_vol(r, config.annualisation),
            "sharpe": annualised_sharpe(r, config.annualisation),
            "skew": skewness(r),
            "kurtosis": kurtosis_non_excess(r),
            "max_drawdown": mdd,
            "VaR": var,
            "CVaR": cvar,
            "total_return": float(nav.iloc[-1] - 1.0),
        })

    return pd.DataFrame(rows)

naive_perf = strategy_performance_table(strategy_returns, config)
naive_perf = naive_perf.merge(strategy_metadata, on="strategy_id", how="left")
naive_perf = naive_perf.sort_values("sharpe", ascending=False)

naive_perf.head(15)

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(naive_perf["sharpe"].dropna(), bins=50)
plt.axvline(naive_perf["sharpe"].iloc[0], linestyle="--", label="max observed Sharpe")
plt.title("Distribution of Observed Strategy Sharpe Ratios")
plt.xlabel("Observed Sharpe")
plt.ylabel("Count")
plt.legend()
plt.show()

## 9. Probabilistic Sharpe Ratio

Compute:

$$
PSR(SR^*=0)
$$

and:

$$
PSR(SR^*=0.5)
$$

for every strategy.

A high observed Sharpe with bad skew, high kurtosis, or short sample gets penalised.

In [ ]:
psr_rows = []

for _, row in naive_perf.iterrows():
    psr_zero = probabilistic_sharpe_ratio(
        row["sharpe"],
        0.0,
        row["n_obs"],
        row["skew"],
        row["kurtosis"]
    )
    psr_threshold = probabilistic_sharpe_ratio(
        row["sharpe"],
        config.sharpe_threshold,
        row["n_obs"],
        row["skew"],
        row["kurtosis"]
    )
    mtrl_zero = minimum_track_record_length(
        row["sharpe"],
        0.0,
        row["skew"],
        row["kurtosis"],
        confidence=config.psr_confidence
    )
    mtrl_threshold = minimum_track_record_length(
        row["sharpe"],
        config.sharpe_threshold,
        row["skew"],
        row["kurtosis"],
        confidence=config.psr_confidence
    )

    psr_rows.append({
        "strategy_id": row["strategy_id"],
        "PSR_gt_0": psr_zero,
        f"PSR_gt_{config.sharpe_threshold}": psr_threshold,
        "min_track_record_len_gt_0": mtrl_zero,
        f"min_track_record_len_gt_{config.sharpe_threshold}": mtrl_threshold,
    })

psr_table = pd.DataFrame(psr_rows)

psr_perf = naive_perf.merge(psr_table, on="strategy_id", how="left")

psr_perf.sort_values(f"PSR_gt_{config.sharpe_threshold}", ascending=False).head(15)

## 10. Expected maximum Sharpe under multiple trials

If many trials are tested, even pure noise produces an impressive maximum.

We estimate:

$$
E[\max(SR)]
$$

from the observed strategy universe.

This becomes the benchmark Sharpe used in the deflated Sharpe calculation.

In [ ]:
n_trials = len(naive_perf)
expected_max_z = expected_max_standard_normal(n_trials)
expected_max_sr = expected_max_sharpe_under_trials(naive_perf["sharpe"], config.n_days)

multiple_testing_report = pd.Series({
    "n_trials": n_trials,
    "expected_max_standard_normal": expected_max_z,
    "mean_observed_sharpe": naive_perf["sharpe"].mean(),
    "std_observed_sharpe": naive_perf["sharpe"].std(ddof=1),
    "expected_max_sharpe_under_trials": expected_max_sr,
    "observed_max_sharpe": naive_perf["sharpe"].max(),
})

multiple_testing_report

## 11. Deflated Sharpe Ratio

Now compute:

$$
DSR = PSR(SR^* = E[\max SR])
$$

This asks:

> Is the observed Sharpe better than what we might expect from the best of many noisy trials?

In [ ]:
dsr_rows = []

for _, row in naive_perf.iterrows():
    dsr = deflated_sharpe_probability(
        row["sharpe"],
        expected_max_sr,
        row["n_obs"],
        row["skew"],
        row["kurtosis"]
    )

    se = sharpe_standard_error(row["sharpe"], row["n_obs"], row["skew"], row["kurtosis"])
    haircut_sr = row["sharpe"] - expected_max_sr

    dsr_rows.append({
        "strategy_id": row["strategy_id"],
        "expected_max_sharpe_benchmark": expected_max_sr,
        "sharpe_standard_error": se,
        "deflated_sharpe_probability": dsr,
        "sharpe_minus_expected_max": haircut_sr,
    })

dsr_table = pd.DataFrame(dsr_rows)

dsr_perf = psr_perf.merge(dsr_table, on="strategy_id", how="left")
dsr_perf = dsr_perf.sort_values("deflated_sharpe_probability", ascending=False)

dsr_perf.head(15)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(dsr_perf["sharpe"], dsr_perf["deflated_sharpe_probability"], alpha=0.65)
plt.axvline(expected_max_sr, linestyle="--", label="expected max Sharpe benchmark")
plt.title("Observed Sharpe vs Deflated Sharpe Probability")
plt.xlabel("Observed Sharpe")
plt.ylabel("Deflated Sharpe probability")
plt.legend()
plt.show()

## 12. False discovery intuition

Because we know which simulated strategies have true alpha, we can check how many top-ranked strategies are genuine.

In real research, we do not know this.

That is exactly why deflation and haircuts matter.

In [ ]:
def top_k_discovery_table(perf_table, sort_col, k_values=(5, 10, 25, 50, 100)):
    rows = []
    sorted_table = perf_table.sort_values(sort_col, ascending=False)

    for k in k_values:
        top = sorted_table.head(k)
        rows.append({
            "ranking_metric": sort_col,
            "top_k": k,
            "true_alpha_count": int(top["has_true_alpha"].sum()),
            "true_alpha_rate": float(top["has_true_alpha"].mean()),
            "mean_sharpe": float(top["sharpe"].mean()),
            "mean_dsr": float(top["deflated_sharpe_probability"].mean()),
        })

    return pd.DataFrame(rows)

discovery_naive = top_k_discovery_table(dsr_perf, "sharpe")
discovery_dsr = top_k_discovery_table(dsr_perf, "deflated_sharpe_probability")

discovery_report = pd.concat([discovery_naive, discovery_dsr], ignore_index=True)

discovery_report

In [ ]:
plt.figure(figsize=(10, 6))
for metric, sub in discovery_report.groupby("ranking_metric"):
    plt.plot(sub["top_k"], sub["true_alpha_rate"], marker="o", label=metric)
plt.title("True Alpha Rate Among Top-K Selected Strategies")
plt.xlabel("Top K")
plt.ylabel("True alpha rate")
plt.legend()
plt.show()

## 13. Performance haircut framework

We build a practical haircut scorecard.

Haircut components:

1. multiple-testing haircut;
2. non-normality haircut;
3. short-history haircut;
4. drawdown haircut;
5. turnover proxy haircut;
6. DSR confidence haircut.

This produces a conservative report Sharpe.

In [ ]:
def performance_haircut(row):
    # Multiple testing: benchmark expected max Sharpe.
    multiple_testing_haircut = max(0.0, row["expected_max_sharpe_benchmark"])

    # Non-normality penalty for bad skew / high kurtosis.
    skew_penalty = max(0.0, -row["skew"]) * 0.10
    kurtosis_penalty = max(0.0, row["kurtosis"] - 3.0) * 0.025
    non_normality_haircut = skew_penalty + kurtosis_penalty

    # Short sample penalty.
    n = row["n_obs"]
    short_sample_haircut = 0.0 if n >= 756 else (756 - n) / 756 * 0.25

    # Drawdown penalty.
    drawdown_haircut = max(0.0, abs(row["max_drawdown"]) - 0.15) * 0.75

    # Turnover proxy from metadata.
    turnover_haircut = row["turnover_assumption"] * 0.10

    # Low DSR confidence penalty.
    dsr_confidence_haircut = max(0.0, 0.95 - row["deflated_sharpe_probability"]) * 0.50

    total_haircut = (
        multiple_testing_haircut
        + non_normality_haircut
        + short_sample_haircut
        + drawdown_haircut
        + turnover_haircut
        + dsr_confidence_haircut
    )

    conservative_sharpe = row["sharpe"] - total_haircut

    return pd.Series({
        "multiple_testing_haircut": multiple_testing_haircut,
        "non_normality_haircut": non_normality_haircut,
        "short_sample_haircut": short_sample_haircut,
        "drawdown_haircut": drawdown_haircut,
        "turnover_haircut": turnover_haircut,
        "dsr_confidence_haircut": dsr_confidence_haircut,
        "total_haircut": total_haircut,
        "conservative_sharpe": conservative_sharpe,
    })

haircut_components = dsr_perf.apply(performance_haircut, axis=1)
haircut_table = pd.concat([dsr_perf.reset_index(drop=True), haircut_components.reset_index(drop=True)], axis=1)
haircut_table = haircut_table.sort_values("conservative_sharpe", ascending=False)

haircut_table.head(15)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(haircut_table["sharpe"], haircut_table["conservative_sharpe"], alpha=0.65)
plt.axline((0, 0), slope=1, linestyle="--")
plt.title("Observed Sharpe vs Conservative Haircut Sharpe")
plt.xlabel("Observed Sharpe")
plt.ylabel("Conservative Sharpe")
plt.show()

## 14. Walk-forward degradation haircut

A practical haircut should also reflect out-of-sample decay.

We simulate a walk-forward process:

1. select top strategies on training windows;
2. evaluate them on subsequent test windows;
3. estimate validation-to-test Sharpe degradation;
4. apply degradation haircut to current research candidates.

In [ ]:
def make_walk_forward_splits(index, train_window=504, test_window=126, step=126):
    rows = []
    start = train_window

    while start + test_window <= len(index):
        rows.append({
            "split_id": len(rows),
            "train_start_idx": start - train_window,
            "train_end_idx": start,
            "test_start_idx": start,
            "test_end_idx": start + test_window,
            "train_start": index[start - train_window],
            "train_end": index[start - 1],
            "test_start": index[start],
            "test_end": index[start + test_window - 1],
        })
        start += step

    return pd.DataFrame(rows)

wf_splits = make_walk_forward_splits(
    strategy_returns.index,
    config.walk_forward_train,
    config.walk_forward_test,
    config.walk_forward_step
)

wf_rows = []

for _, split in wf_splits.iterrows():
    train = strategy_returns.iloc[int(split["train_start_idx"]):int(split["train_end_idx"])]
    test = strategy_returns.iloc[int(split["test_start_idx"]):int(split["test_end_idx"])]

    train_perf = strategy_performance_table(train, config).sort_values("sharpe", ascending=False)
    top_train = train_perf.head(10)["strategy_id"].tolist()

    for strategy_id in top_train:
        train_metrics = train_perf[train_perf["strategy_id"] == strategy_id].iloc[0].to_dict()
        test_metrics = strategy_metrics(test[strategy_id], config)

        wf_rows.append({
            "split_id": int(split["split_id"]),
            "strategy_id": strategy_id,
            "train_sharpe": train_metrics["sharpe"],
            "test_sharpe": test_metrics["sharpe_proxy"],
            "sharpe_degradation": test_metrics["sharpe_proxy"] - train_metrics["sharpe"],
            "train_return": train_metrics["annualised_return"],
            "test_return": test_metrics["annualised_return"],
            "return_degradation": test_metrics["annualised_return"] - train_metrics["annualised_return"],
            "test_CVaR": test_metrics["CVaR"],
            "test_max_drawdown": test_metrics["max_drawdown"],
        })

wf_degradation = pd.DataFrame(wf_rows)

wf_degradation_summary = wf_degradation[["sharpe_degradation", "return_degradation"]].agg(["mean", "median", "std", "min", "max"]).T

wf_degradation_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(wf_degradation["sharpe_degradation"], bins=30)
plt.axvline(0, linestyle="--")
plt.title("Walk-Forward Train-to-Test Sharpe Degradation")
plt.xlabel("Test Sharpe - Train Sharpe")
plt.ylabel("Count")
plt.show()

## 15. Apply walk-forward degradation haircut

A conservative approach:

$$
\begin{aligned}
SR_{final} &= SR_{conservative} \\
&\quad + median(Degradation)
\end{aligned}
$$

If median degradation is negative, it reduces the reported Sharpe further.

In [ ]:
median_wf_sharpe_degradation = wf_degradation["sharpe_degradation"].median()

haircut_table["walk_forward_degradation_haircut"] = min(0.0, median_wf_sharpe_degradation)
haircut_table["final_report_sharpe"] = haircut_table["conservative_sharpe"] + haircut_table["walk_forward_degradation_haircut"]

final_ranked = haircut_table.sort_values("final_report_sharpe", ascending=False)

final_ranked.head(15)

## 16. Report-card for selected strategy

Select the top final-report Sharpe strategy and produce a research report card.

In [ ]:
selected_strategy = final_ranked.iloc[0]["strategy_id"]
selected_returns = strategy_returns[selected_strategy]
selected_nav = (1 + selected_returns).cumprod()

selected_report = final_ranked[final_ranked["strategy_id"] == selected_strategy].iloc[0].to_dict()

selected_report_card = pd.DataFrame([{
    "selected_strategy": selected_strategy,
    "observed_sharpe": selected_report["sharpe"],
    "PSR_gt_0": selected_report["PSR_gt_0"],
    f"PSR_gt_{config.sharpe_threshold}": selected_report[f"PSR_gt_{config.sharpe_threshold}"],
    "expected_max_sharpe_benchmark": selected_report["expected_max_sharpe_benchmark"],
    "deflated_sharpe_probability": selected_report["deflated_sharpe_probability"],
    "total_haircut": selected_report["total_haircut"],
    "conservative_sharpe": selected_report["conservative_sharpe"],
    "walk_forward_degradation_haircut": selected_report["walk_forward_degradation_haircut"],
    "final_report_sharpe": selected_report["final_report_sharpe"],
    "observed_annualised_return": selected_report["annualised_return"],
    "observed_annualised_vol": selected_report["annualised_vol"],
    "max_drawdown": selected_report["max_drawdown"],
    "CVaR": selected_report["CVaR"],
    "has_true_alpha_in_simulation": bool(selected_report["has_true_alpha"]),
}])

selected_report_card

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(selected_nav.index, selected_nav)
plt.title(f"Selected Strategy Equity Curve: {selected_strategy}")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.show()

dd, _ = max_drawdown(selected_nav)
plt.figure(figsize=(12, 5))
plt.plot(dd.index, dd)
plt.title("Selected Strategy Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.show()

## 17. Stress-regime performance

A strategy with a strong Sharpe can still be unacceptable if it fails during stress.

In [ ]:
def regime_performance(return_series, regime_info, config):
    joined = pd.concat([return_series.rename("return"), regime_info], axis=1).dropna()
    rows = []

    for bucket, group in joined.groupby("stress_type"):
        if len(group) < 5:
            continue
        metrics = strategy_metrics(group["return"], config)
        rows.append({
            "bucket": bucket,
            "n_obs": len(group),
            **metrics,
        })

    for bucket, group in joined.groupby("vol_regime"):
        if len(group) < 5:
            continue
        metrics = strategy_metrics(group["return"], config)
        rows.append({
            "bucket": f"vol_regime_{bucket}",
            "n_obs": len(group),
            **metrics,
        })

    return pd.DataFrame(rows)

selected_regime_perf = regime_performance(selected_returns, regime_info, config)

selected_regime_perf

## 18. Benchmark against naive selection

Compare the selected final-report strategy to:

1. naive top Sharpe;
2. top DSR;
3. top conservative haircut;
4. equal-weight ensemble of all trials.

This shows how selection criteria change the result.

In [ ]:
naive_top = naive_perf.iloc[0]["strategy_id"]
dsr_top = dsr_perf.iloc[0]["strategy_id"]
haircut_top = haircut_table.iloc[0]["strategy_id"]

comparison_returns = pd.DataFrame({
    "naive_top_sharpe": strategy_returns[naive_top],
    "top_dsr": strategy_returns[dsr_top],
    "top_haircut": strategy_returns[haircut_top],
    "top_final_report": strategy_returns[selected_strategy],
    "equal_trial_ensemble": strategy_returns.mean(axis=1),
})

selection_comparison = pd.DataFrame([
    {"selection": col, **strategy_metrics(comparison_returns[col], config)}
    for col in comparison_returns.columns
]).sort_values("sharpe_proxy", ascending=False)

selection_comparison

In [ ]:
plt.figure(figsize=(12, 6))
for col in comparison_returns.columns:
    plt.plot(comparison_returns.index, (1 + comparison_returns[col]).cumprod(), label=col, alpha=0.85)
plt.title("Selection Method Comparison")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

## 19. Governance flags

A performance report should be reviewed if:

1. DSR is low;
2. final report Sharpe is much lower than observed Sharpe;
3. expected max Sharpe under trials is close to observed Sharpe;
4. walk-forward degradation is severe;
5. strategy has bad skew or high kurtosis;
6. drawdown or CVaR is unacceptable;
7. the result depends on naive selection.

In [ ]:
governance_flags = pd.DataFrame([{
    "selected_strategy": selected_strategy,
    "observed_sharpe": selected_report["sharpe"],
    "final_report_sharpe": selected_report["final_report_sharpe"],
    "deflated_sharpe_probability": selected_report["deflated_sharpe_probability"],
    "expected_max_sharpe_benchmark": selected_report["expected_max_sharpe_benchmark"],
    "total_haircut": selected_report["total_haircut"],
    "median_walk_forward_sharpe_degradation": median_wf_sharpe_degradation,
    "skew": selected_report["skew"],
    "kurtosis": selected_report["kurtosis"],
    "max_drawdown": selected_report["max_drawdown"],
    "CVaR": selected_report["CVaR"],
    "flag_dsr_below_95pct": bool(selected_report["deflated_sharpe_probability"] < 0.95),
    "flag_final_sharpe_below_0_5": bool(selected_report["final_report_sharpe"] < 0.50),
    "flag_observed_to_final_sharpe_gap_above_1": bool(selected_report["sharpe"] - selected_report["final_report_sharpe"] > 1.0),
    "flag_expected_max_close_to_observed": bool(selected_report["sharpe"] - selected_report["expected_max_sharpe_benchmark"] < 0.25),
    "flag_walk_forward_degradation_below_minus_0_75": bool(median_wf_sharpe_degradation < -0.75),
    "flag_bad_skew": bool(selected_report["skew"] < -1.0),
    "flag_high_kurtosis": bool(selected_report["kurtosis"] > 8.0),
    "flag_drawdown_below_minus_25pct": bool(selected_report["max_drawdown"] < -0.25),
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_dsr_below_95pct",
        "flag_final_sharpe_below_0_5",
        "flag_observed_to_final_sharpe_gap_above_1",
        "flag_expected_max_close_to_observed",
        "flag_walk_forward_degradation_below_minus_0_75",
        "flag_bad_skew",
        "flag_high_kurtosis",
        "flag_drawdown_below_minus_25pct",
    ]
].any(axis=1)

governance_flags

## 20. Audit checks

Numerical and process checks:

1. Sharpe values are finite;
2. PSR and DSR are probabilities;
3. expected max Sharpe benchmark is finite;
4. haircuts are non-negative except degradation adjustment;
5. final report Sharpe does not exceed observed Sharpe unless walk-forward degradation is positive;
6. walk-forward splits are chronological;
7. saved selected strategy exists in the strategy universe.

In [ ]:
audit_rows = []

audit_rows.append({
    "check": "sharpe_values_finite",
    "value": bool(np.isfinite(naive_perf["sharpe"]).all()),
    "passed": bool(np.isfinite(naive_perf["sharpe"]).all()),
})

prob_cols = ["PSR_gt_0", f"PSR_gt_{config.sharpe_threshold}", "deflated_sharpe_probability"]
probabilities_valid = bool(((dsr_perf[prob_cols] >= 0) & (dsr_perf[prob_cols] <= 1)).all().all())
audit_rows.append({
    "check": "psr_dsr_probabilities_in_unit_interval",
    "value": probabilities_valid,
    "passed": probabilities_valid,
})

audit_rows.append({
    "check": "expected_max_sharpe_finite",
    "value": float(expected_max_sr),
    "passed": bool(np.isfinite(expected_max_sr)),
})

haircuts_nonnegative = bool((haircut_table[[
    "multiple_testing_haircut",
    "non_normality_haircut",
    "short_sample_haircut",
    "drawdown_haircut",
    "turnover_haircut",
    "dsr_confidence_haircut",
    "total_haircut"
]] >= -1e-12).all().all())
audit_rows.append({
    "check": "haircuts_nonnegative",
    "value": haircuts_nonnegative,
    "passed": haircuts_nonnegative,
})

final_leq_observed = bool((final_ranked["final_report_sharpe"] <= final_ranked["sharpe"] + 1e-12).all())
audit_rows.append({
    "check": "final_report_sharpe_not_above_observed",
    "value": final_leq_observed,
    "passed": final_leq_observed,
})

wf_chronological = bool((wf_splits["train_end"] < wf_splits["test_start"]).all())
audit_rows.append({
    "check": "walk_forward_splits_chronological",
    "value": wf_chronological,
    "passed": wf_chronological,
})

selected_exists = selected_strategy in strategy_returns.columns
audit_rows.append({
    "check": "selected_strategy_exists",
    "value": selected_exists,
    "passed": selected_exists,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 21. Practical checklist for deflated performance reporting

Before reporting a Sharpe ratio:

1. **Report number of trials**  
   A Sharpe from 1 trial and 500 trials are not the same evidence.

2. **Report skew and kurtosis**  
   Non-normal returns make Sharpe uncertainty worse.

3. **Compute PSR**  
   What is the probability Sharpe exceeds zero or a threshold?

4. **Compute DSR**  
   Does performance survive multiple-testing adjustment?

5. **Estimate expected max Sharpe under noise**  
   What Sharpe might the best random strategy achieve?

6. **Apply performance haircut**  
   Report conservative Sharpe, not just observed Sharpe.

7. **Include walk-forward degradation**  
   Hindsight winners usually decay.

8. **Report drawdown and CVaR**  
   Sharpe alone hides tail risk.

9. **Keep failed trials**  
   Unreported failed experiments create selection bias.

10. **Save the research audit trail**  
   Performance claims should be reproducible.

## 22. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed/performance_haircut_and_deflated_sharpe_v1")
output_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "strategy_returns": output_dir / "strategy_returns.csv",
    "strategy_metadata": output_dir / "strategy_metadata.csv",
    "regime_info": output_dir / "regime_info.csv",
    "naive_performance": output_dir / "naive_performance.csv",
    "psr_table": output_dir / "psr_table.csv",
    "multiple_testing_report": output_dir / "multiple_testing_report.csv",
    "dsr_table": output_dir / "dsr_table.csv",
    "dsr_performance": output_dir / "dsr_performance.csv",
    "discovery_report": output_dir / "discovery_report.csv",
    "haircut_table": output_dir / "haircut_table.csv",
    "walk_forward_degradation": output_dir / "walk_forward_degradation.csv",
    "walk_forward_degradation_summary": output_dir / "walk_forward_degradation_summary.csv",
    "final_ranked": output_dir / "final_ranked.csv",
    "selected_report_card": output_dir / "selected_report_card.csv",
    "selected_regime_performance": output_dir / "selected_regime_performance.csv",
    "selection_comparison": output_dir / "selection_comparison.csv",
    "governance_flags": output_dir / "governance_flags.csv",
    "audit_report": output_dir / "audit_report.csv",
    "manifest": output_dir / "manifest.json",
}

strategy_returns.to_csv(paths["strategy_returns"])
strategy_metadata.to_csv(paths["strategy_metadata"], index=False)
regime_info.to_csv(paths["regime_info"])
naive_perf.to_csv(paths["naive_performance"], index=False)
psr_table.to_csv(paths["psr_table"], index=False)
multiple_testing_report.to_frame("value").to_csv(paths["multiple_testing_report"])
dsr_table.to_csv(paths["dsr_table"], index=False)
dsr_perf.to_csv(paths["dsr_performance"], index=False)
discovery_report.to_csv(paths["discovery_report"], index=False)
haircut_table.to_csv(paths["haircut_table"], index=False)
wf_degradation.to_csv(paths["walk_forward_degradation"], index=False)
wf_degradation_summary.to_csv(paths["walk_forward_degradation_summary"])
final_ranked.to_csv(paths["final_ranked"], index=False)
selected_report_card.to_csv(paths["selected_report_card"], index=False)
selected_regime_perf.to_csv(paths["selected_regime_performance"], index=False)
selection_comparison.to_csv(paths["selection_comparison"], index=False)
governance_flags.to_csv(paths["governance_flags"], index=False)
audit_report.to_csv(paths["audit_report"], index=False)

manifest = {
    "dataset_name": "performance_haircut_and_deflated_sharpe_outputs",
    "schema_version": "performance_haircut_and_deflated_sharpe_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "n_strategy_trials": config.n_strategy_trials,
    "n_true_alpha_strategies": config.n_true_alpha_strategies,
    "expected_max_sharpe_under_trials": float(expected_max_sr),
    "observed_max_sharpe": float(naive_perf["sharpe"].max()),
    "selected_strategy": selected_strategy,
    "selected_report_card": selected_report_card.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "Probabilistic Sharpe Ratio is computed using skew and kurtosis adjustment.",
        "Deflated Sharpe Probability uses an expected maximum Sharpe benchmark from multiple trials.",
        "Performance haircut includes multiple testing, non-normality, short sample, drawdown, turnover and DSR confidence penalties.",
        "Walk-forward degradation is estimated from top training strategies applied to future test windows.",
        "The simulation records true-alpha labels only for educational diagnostics; real research does not know these labels.",
        "This notebook is about conservative performance reporting, not proving strategy profitability."
    ],
}

paths["manifest"].write_text(json.dumps(manifest, indent=2, default=str))

paths["final_ranked"], paths["selected_report_card"], paths["governance_flags"], paths["audit_report"], paths["manifest"]

## 23. Limitations

### Simplified DSR approximation

The notebook uses a practical approximation for the expected maximum Sharpe benchmark. Production-grade DSR should carefully estimate the number of independent trials and strategy correlation.

### Synthetic true-alpha labels

The simulation knows which strategies have true alpha. Real researchers do not.

### Strategy dependence ignored

Many strategy variants are correlated. Counting every grid point as fully independent can overstate the multiple-testing penalty.

### Normal approximation

PSR/DSR uses asymptotic Sharpe uncertainty approximations. Short samples and extreme tails can still break assumptions.

### Haircut is partly judgemental

A performance haircut is a governance tool, not a universal mathematical truth.

### No full White Reality Check

This notebook does not implement White's Reality Check or SPA tests.

### No complete research audit database

Later notebooks should track every experiment, code version, dataset version and selection decision.

## 24. Research frontier and extensions

Important extensions include:

1. Deflated Sharpe Ratio with correlated trials;
2. Probabilistic Sharpe with autocorrelation correction;
3. White's Reality Check;
4. Hansen's Superior Predictive Ability test;
5. False Discovery Rate control;
6. Probability of Backtest Overfitting;
7. Combinatorially Symmetric Cross-Validation;
8. Bayesian posterior over Sharpe;
9. research-trial database and experiment registry;
10. live shadow-mode degradation tracking.

## 25. Suggested follow-up notebooks

This notebook naturally leads to:

1. **05_07_parameter_sensitivity_and_overfitting**  
   Study parameter stability and overfit surfaces.

2. **05_08_strategy_capacity_and_market_impact**  
   Add capacity haircuts to performance.

3. **05_09_purged_cross_validation_backtesting**  
   Validate overlapping-label strategies.

4. **05_10_research_audit_trail_and_manifest**  
   Record every research trial.

5. **05_11_live_shadow_mode_monitoring**  
   Track live performance decay.

6. **07_06_research_governance_capstone**  
   Combine DSR, PBO, Bayesian calibration and audit trails.

## 26. Summary

This notebook implemented performance haircut and deflated Sharpe analysis.

It showed how to:

1. simulate many strategy trials;
2. compute naive performance metrics;
3. compute Probabilistic Sharpe Ratio;
4. estimate minimum track record length;
5. estimate expected maximum Sharpe under many trials;
6. compute Deflated Sharpe probability;
7. inspect false discovery intuition;
8. create a transparent performance haircut;
9. estimate walk-forward degradation;
10. produce a final report Sharpe;
11. compare naive, DSR, haircut and ensemble selection;
12. analyse stress-regime performance;
13. create governance flags and audit checks;
14. save outputs and manifest.

The key computational takeaway:

> The Sharpe ratio must be interpreted through sample length, skew, kurtosis, and the number of trials searched.

The key financial takeaway:

> If you tried hundreds of variants, your best Sharpe is guilty until deflated.

## 27. Further reading

- Bailey and López de Prado on the Deflated Sharpe Ratio.
- Bailey et al. on Probability of Backtest Overfitting.
- López de Prado, *Advances in Financial Machine Learning.*
- White's Reality Check.
- Hansen's Superior Predictive Ability test.
- Harvey, Liu and Zhu on multiple testing in factor research.
- Institutional model-risk literature on performance haircuts and research governance.